# Notes

This notebook handles the processing of ground-truth spectra from the spectrometers. It does not require anything else than the data.

# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import config
from pathlib import Path
from matplotlib import pyplot as plt
import pandas as pd
import json
import numpy as np
import os
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union
import io
import re

import hatfield_funcs as hf
import hatfield_funcs.helpers as hfh

# project root = one level up from notebooks/ and potentially more form here
project_root = Path("../..").resolve()
sys.path.insert(0, str(project_root / "src"))

from my_extra_to_hatfield_funcs.io_funcs import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
if os.getenv("GA_STORAGE_ACCOUNT_NAME") is not None:
    # this is what gets used when using Azure cloud storage (GeoAnalytics)
    hf.config.configure_azure()
    hf.config.configure_data_root_path(config.CONTAINER_NAME)
else:
    # root path defined for local data
    raise Exception(
        "This notebook is configured to run in the cloud. Please set up your Azure storage account and container name."
    )


In [25]:
plt.style.use("default")
%config InlineBackend.figure_format = 'retina'

# Spectra Processing

## Classes and Functions

In [37]:
@dataclass(frozen=True)
class PanelCurve:
    wavelength_nm: np.ndarray
    reflectance: np.ndarray

def interp_panel_to_wavelengths(
    panel: PanelCurve, target_wl_nm: np.ndarray
) -> np.ndarray:
    """Linear interpolation with NaN outside range."""
    return np.interp(
        target_wl_nm, panel.wavelength_nm, panel.reflectance, left=np.nan, right=np.nan
    )


In [ ]:
def parse_degmin(s: str) -> Optional[float]:
    """Parse ddmm.mmmmN or dddmm.mmmmW into decimal degrees."""
    if s is None:
        return None
    txt = str(s).strip()
    if txt == "":
        return None

    m = re.match(r"^\s*([0-9]+\.?[0-9]*)\s*([NSEW])\s*$", txt, re.IGNORECASE)
    if not m:
        return None
    val = float(m.group(1))
    hemi = m.group(2).upper()

    # Determine deg digits by hemisphere type: lat typically 2 deg digits, lon typically 3
    # But we can infer by total digits before decimal: if >= 5 -> lon
    digits = re.sub(r"\D", "", m.group(1))
    if len(digits) >= 5:
        deg_digits = 3
    else:
        deg_digits = 2

    deg = int(val // 100)
    minutes = val - deg * 100
    dec = deg + minutes / 60.0

    if hemi in ("S", "W"):
        dec = -dec
    return dec


In [30]:
def _as_list(x):
    return x if isinstance(x, list) else [x]

def _side_value(meta: Dict, key: str, side: str):
    # side: 'ref' or 'tgt'
    v = meta.get(key)
    if v is None:
        return None
    if isinstance(v, list) and len(v) >= 2:
        return v[0] if side == "ref" else v[1]
    return v

def _parse_time(ts) -> Optional[pd.Timestamp]:
    if ts is None or (isinstance(ts, float) and np.isnan(ts)):
        return None
    s = str(ts).strip()
    if s == "":
        return None
    # Example: 05/19/2026 06:09:35PM
    return pd.to_datetime(s, errors="coerce")


def summarize_by_group_stat_rows(
    refl_df: pd.DataFrame, group_cols: list[str], wl_cols: list[str]
) -> pd.DataFrame:
    """
    Return a summary table with two rows per group:
      stat='mean' and stat='std'
    keeping wavelength columns as wl_* so to_spectral_matrix() works.
    """
    g = refl_df.groupby(group_cols, dropna=False)

    n_scans = g.size().rename("n_scans")
    mean_df = g[wl_cols].mean()
    std_df = g[wl_cols].std()

    mean_out = mean_df.reset_index()
    mean_out["stat"] = "mean"
    mean_out = mean_out.merge(n_scans.reset_index(), on=group_cols, how="left")

    std_out = std_df.reset_index()
    std_out["stat"] = "std"
    std_out = std_out.merge(n_scans.reset_index(), on=group_cols, how="left")

    # Keep a consistent column order
    return pd.concat([mean_out, std_out], ignore_index=True)[
        group_cols + ["stat", "n_scans"] + wl_cols
    ]

In [ ]:
def _parse_list(value: Any) -> List[int]:
    if value is None or pd.isna(value):
        return []
    if isinstance(value, list):
        out = []
        for v in value:
            s = str(v).strip()
            if s != "":
                out.append(int(float(s)))
        return out
    s = str(value).strip()
    if s == "":
        return []
    return [int(float(x.strip())) for x in s.split(",") if x.strip()]


def _parse_single(value: Any) -> Optional[int]:
    if value is None or pd.isna(value):
        return None
    if isinstance(value, list):
        for v in value:
            s = str(v).strip()
            if s != "":
                return int(float(s))
        return None
    s = str(value).strip()
    if s == "":
        return None
    if "," in s:
        s = next((x.strip() for x in s.split(",") if x.strip()), "")
        if s == "":
            return None
    return int(float(s))


In [27]:
@dataclass
class SigFile:
    metadata: Dict[str, Any]
    data: List[List[float]]
    columns: List[str]


class SigReader:
    """Parse `.sig` files from local or Azure Blob Storage.

    Files are expected to contain a ``key=value`` header followed by a
    ``data=`` marker and a whitespace-delimited numeric table.
    """

    def read(
        self,
        path: Union[str, Path],
        *,
        expected_cols: Optional[int] = None,
    ) -> SigFile:
        # Do not convert the path to pathlib.Path because Azure paths are
        # not local filesystem paths.
        full_path = hfh.return_full_path(str(path))
        fs = hf.config.get_filesystem()

        metadata: Dict[str, Any] = {}
        data_rows: List[List[float]] = []
        in_data = False

        try:
            # Open as binary and wrap as text. This is reliable for both
            # local fsspec filesystems and AzureBlobFileSystem.
            with fs.open(full_path, "rb") as binary_file:
                with io.TextIOWrapper(
                    binary_file,
                    encoding="utf-8",
                    errors="replace",
                ) as file:
                    for raw_line in file:
                        line = raw_line.strip()

                        if not line or line.startswith("/***"):
                            continue

                        if not in_data:
                            if line.lower() == "data=":
                                in_data = True
                                continue

                            if "=" not in line:
                                continue

                            key, value = line.split("=", 1)
                            key = key.strip()
                            value = value.strip()

                            parsed = self._parse_header_value(value)

                            if key in metadata:
                                if not isinstance(metadata[key], list):
                                    metadata[key] = [metadata[key]]

                                metadata[key].append(parsed)
                            else:
                                metadata[key] = parsed

                        else:
                            parts = line.split()

                            if (
                                expected_cols is not None
                                and len(parts) != expected_cols
                            ):
                                continue

                            try:
                                row = [float(part) for part in parts]
                            except ValueError:
                                continue

                            data_rows.append(row)

        except Exception as error:
            raise OSError(f"Error reading SIG file {full_path!r}") from error

        ncols = (
            expected_cols
            if expected_cols is not None
            else len(data_rows[0])
            if data_rows
            else 0
        )

        columns = self._default_columns(ncols)

        return SigFile(
            metadata=metadata,
            data=data_rows,
            columns=columns,
        )

    def _default_columns(self, ncols: int) -> List[str]:
        if ncols == 4:
            return ["wavelength_nm", "reference", "target", "relative_reflectance"]
        if ncols == 3:
            return ["wavelength_nm", "reference", "target"]
        if ncols == 2:
            return ["wavelength_nm", "value"]
        return [f"col_{i + 1}" for i in range(ncols)]

    def _parse_header_value(self, value: str) -> Any:
        if value == "":
            return ""
        if value.startswith("{") and value.endswith("}"):
            try:
                return json.loads(value)
            except Exception:
                return value
        if "," in value:
            items = [x.strip() for x in value.split(",")]
            return [self._parse_scalar(x) for x in items]
        return self._parse_scalar(value)

    def _parse_scalar(self, token: str) -> Any:
        if token == "":
            return ""
        # int
        try:
            if token.isdigit() or (token.startswith("-") and token[1:].isdigit()):
                return int(token)
        except Exception:
            pass
        # float
        try:
            return float(token)
        except Exception:
            return token


def read_json_from_file(file_path):
    """
    Read a JSON file into a Python object from the configured filesystem (local or Azure Blob Storage).
    Parameters
    ----------
    file_path : str
        The path to the JSON file, with or without storage container/root.
    Returns
    -------
    object
        The Python object containing the data from the JSON file.
    """
    try:
        file_path = hfh.return_full_path(file_path)
        fs = hf.config.get_filesystem()
        with fs.open(file_path, "r") as f:
            data = json.load(f)
        return data
    except Exception as e:
        print(f"Error reading JSON file: {e}")
        raise


def write_df_to_parquet(
    df: pd.DataFrame, file_path: str, compression: str = "zstd", **kwargs
):
    """
    Write a pandas DataFrame to a Parquet file in the configured filesystem (local or Azure Blob Storage).
    Parameters
    ----------
    df : pd.DataFrame
        The DataFrame to write.
    file_path : str
        The path to the Parquet file, with or without storage container/root.
    compression : str, optional
        The compression algorithm to use for the Parquet file (default is "zstd").
    """
    try:
        file_path = hfh.return_full_path(file_path)
        fs = hf.config.get_filesystem()
        with fs.open(file_path, "wb") as f:
            df.to_parquet(f, compression=compression, **kwargs)
    except Exception as e:
        print(f"Error writing DataFrame to Parquet: {e}")
        raise


def create_dir_if_not_exists(dir_path):
    try:
        dir_path = hfh.return_full_path(dir_path)
        fs = hf.config.get_filesystem()
        if not fs.exists(dir_path):
            fs.makedirs(dir_path)
    except Exception as e:
        print(f"Error creating directory: {e}")
        raise


# Create a function to check if file exists in the configured filesystem (local or Azure Blob Storage)
def file_exists(file_path):
    try:
        file_path = hfh.return_full_path(file_path)
        fs = hf.config.get_filesystem()
        return fs.exists(file_path)
    except Exception as e:
        print(f"Error checking file existence: {e}")
        raise


def load_station_metadata(path: Path) -> pd.DataFrame:
    # geojson = json.loads(Path(path).read_text(encoding="utf-8"))
    geojson = read_json_from_file(hfh.return_full_path(path))
    df = pd.json_normalize([f["properties"] for f in geojson["features"]])

    # expected fields (names based on your previous code)
    if "first_target_scan_id" in df.columns:
        df["first_scan_id"] = df["first_target_scan_id"].apply(_parse_single)
    if "last_target_scan_id" in df.columns:
        df["last_scan_id"] = df["last_target_scan_id"].apply(_parse_single)

    if "ref_panel_target_scan_ids" in df.columns:
        df["tp_scan_ids"] = df["ref_panel_target_scan_ids"].apply(_parse_list)
    else:
        df["tp_scan_ids"] = [[] for _ in range(len(df))]

    if "target_scan_ids_to_ignore" in df.columns:
        df["ignore_scan_ids"] = df["target_scan_ids_to_ignore"].apply(_parse_list)
    else:
        df["ignore_scan_ids"] = [[] for _ in range(len(df))]

    # datetime used for HR.MMDDYY prefix
    dt_col = (
        "date_and_time_local"
        if "date_and_time_local" in df.columns
        else ("date_and_time" if "date_and_time" in df.columns else None)
    )
    if dt_col is None:
        raise ValueError(
            "Metadata is missing date/time field needed for SIG filename prefix (date_and_time_local/date_and_time)"
        )

    df["acq_datetime"] = pd.to_datetime(df[dt_col], errors="coerce")
    if df["acq_datetime"].isna().any():
        # allow but warn at runtime; for now, error
        raise ValueError(
            "Some rows have invalid acquisition datetime in station metadata"
        )

    # station id and round
    if "station_name" not in df.columns:
        raise ValueError("Metadata missing station_name")
    if "round" not in df.columns:
        raise ValueError("Metadata missing round")

    df["station_id"] = df["station_name"].astype(str)
    df["round"] = df["round"].astype(int)

    return df[
        [
            "station_id",
            "round",
            "first_scan_id",
            "last_scan_id",
            "tp_scan_ids",
            "ignore_scan_ids",
            "acq_datetime",
        ]
    ]


def wl_to_col(wl_nm: float, fmt: dict) -> str:
    # fixed-width zero padded, fixed decimals; safe lexical sorting
    num = f"{wl_nm:0{fmt['width']}.{fmt['decimals']}f}"
    return f"{fmt['prefix']}{num}"


def make_wl_columns(wavelengths_nm: np.ndarray, fmt: dict) -> List[str]:
    return [wl_to_col(float(w), fmt) for w in wavelengths_nm]


## Do

In [28]:
station_df = load_station_metadata(
    config.CONTAINER_FILE_PATHS["station_metadata_geojson"]
)


In [38]:
out_dir = config.CONTAINER_DIR_PATHS["processed_spectra"]

In [40]:
for site in config.SITES:
    spectra_dir = Path(config.CONTAINER_DIR_PATHS["raw_spectra_per_site"][site])

    panel_path = Path(config.CONTAINER_FILE_PATHS["white_panel_calibration_txt_per_site"][site])
    # Get panel calibration curve
    panel = read_df_from_csv(panel_path, sep="\t", header=None)
    panel.index.name = "wavelength_nm"
    panel.columns = ["reflectance"]
    panel.reset_index(inplace=True)
    panel.head()

    seen_ref_times_by_group = {}

    reader = SigReader()

    # Filter station metadata by station_id prefix: W* or B*
    if site == "westham":
        # Station starts by W or is 'grey' or 'black'
        st = station_df[
            station_df["station_id"].str.startswith("W")
            | station_df["station_id"].isin(["grey", "black"])
        ].copy()
    elif site == "brunswick":
        st = station_df[station_df["station_id"].str.startswith("B")].copy()
    else:
        raise ValueError(f"Unknown site {site}")

    if st.empty:
        raise ValueError(f"No stations found for site {site}")

    radiance_rows = []
    target_rows_native = []

    native_wl = None

    # Collect reference records for reflectance interpolation
    # We'll store per station+round references separately.
    references_by_group = {}
    targets_by_group = {}

    for i_row, row in enumerate(st.itertuples(index=False)):
        station_id = row.station_id
        round_num = int(row.round)
        first_scan = int(row.first_scan_id)
        last_scan = int(row.last_scan_id)
        tp_scan_ids = set(row.tp_scan_ids or [])
        ignore_scan_ids = set(row.ignore_scan_ids or [])
        acq_dt = pd.Timestamp(row.acq_datetime)

        # SIG file prefix uses local date in file naming: HR.MMDDYY.
        prefix = f"HR.{acq_dt.strftime('%m%d%y')}."

        group_key = (station_id, round_num)
        seen_ref_times_by_group.setdefault(group_key, set())
        references_by_group.setdefault(group_key, [])
        targets_by_group.setdefault(group_key, [])

        for scan_id in range(first_scan, last_scan + 1):
            if scan_id in ignore_scan_ids:
                continue
            sig_path = Path(spectra_dir) / f"{prefix}{scan_id:04d}.sig"
            if not file_exists(sig_path):
                raise FileNotFoundError(f"Missing SIG file: {sig_path}")
            sig = reader.read(sig_path, expected_cols=4)
            df = pd.DataFrame(sig.data, columns=sig.columns)

            wl = df["wavelength_nm"].to_numpy(dtype=float)
            if native_wl is None:
                native_wl = wl
            else:
                # strict check; adjust to allclose if needed later
                if wl.shape != native_wl.shape or not np.allclose(
                    wl, native_wl, rtol=0, atol=1e-6
                ):
                    raise ValueError(
                        f"Wavelength mismatch within site {site} at {sig_path.name}"
                    )

            # times (paired)
            times = _as_list(sig.metadata.get("time", []))
            ref_time = _parse_time(times[0] if len(times) > 0 else None)
            tgt_time = _parse_time(times[1] if len(times) > 1 else None)

            # metadata fields (paired vs global)
            lat_ref = parse_degmin(_side_value(sig.metadata, "latitude", "ref"))
            lon_ref = parse_degmin(_side_value(sig.metadata, "longitude", "ref"))
            lat_tgt = parse_degmin(_side_value(sig.metadata, "latitude", "tgt"))
            lon_tgt = parse_degmin(_side_value(sig.metadata, "longitude", "tgt"))

            sunz_ref = _side_value(sig.metadata, "sun zenith", "ref")
            sunz_tgt = _side_value(sig.metadata, "sun zenith", "tgt")
            suna_ref = _side_value(sig.metadata, "sun azimuth", "ref")
            suna_tgt = _side_value(sig.metadata, "sun azimuth", "tgt")

            scan_method = sig.metadata.get("scan method")
            scan_time_ref = _side_value(sig.metadata, "scan time", "ref")
            scan_time_tgt = _side_value(sig.metadata, "scan time", "tgt")
            scan_coadds_ref = _side_value(sig.metadata, "scan coadds", "ref")
            scan_coadds_tgt = _side_value(sig.metadata, "scan coadds", "tgt")

            # Radiance vectors
            rad_ref = df["reference"].to_numpy(dtype=float)
            rad_tgt = df["target"].to_numpy(dtype=float)

            # Normalize time to avoid tiny parsing differences (optional but helpful)
            ref_time_key = (
                None if ref_time is None else pd.Timestamp(ref_time).floor("s")
            )

            if ref_time_key not in seen_ref_times_by_group[group_key]:
                seen_ref_times_by_group[group_key].add(ref_time_key)

                # Build radiance output rows: one R + one target row (T or Tp)
                radiance_rows.append(
                    {
                        "site": site,
                        "station_id": station_id,
                        "round": round_num,
                        "scan_id": scan_id,
                        "scan_type": "R",
                        "sig_file": sig_path.name,
                        "acquisition_time_local": ref_time,
                        "latitude_deg": lat_ref,
                        "longitude_deg": lon_ref,
                        "sun_zenith_deg": float(sunz_ref)
                        if sunz_ref is not None
                        else np.nan,
                        "sun_azimuth_deg": float(suna_ref)
                        if suna_ref is not None
                        else np.nan,
                        "scan_method": scan_method[0]
                        if isinstance(scan_method, list) and scan_method
                        else (scan_method if scan_method is not None else None),
                        "scan_time_s": float(scan_time_ref)
                        if scan_time_ref is not None
                        else np.nan,
                        "scan_coadds": float(scan_coadds_ref)
                        if scan_coadds_ref is not None
                        else np.nan,
                        "_vector": rad_ref,
                    }
                )

            target_type = "Tp" if scan_id in tp_scan_ids else "T"
            radiance_rows.append(
                {
                    "site": site,
                    "station_id": station_id,
                    "round": round_num,
                    "scan_id": scan_id,
                    "scan_type": target_type,
                    "sig_file": sig_path.name,
                    "acquisition_time_local": tgt_time,
                    "latitude_deg": lat_tgt,
                    "longitude_deg": lon_tgt,
                    "sun_zenith_deg": float(sunz_tgt)
                    if sunz_tgt is not None
                    else np.nan,
                    "sun_azimuth_deg": float(suna_tgt)
                    if suna_tgt is not None
                    else np.nan,
                    "scan_method": scan_method[1]
                    if isinstance(scan_method, list) and len(scan_method) > 1
                    else (scan_method if scan_method is not None else None),
                    "scan_time_s": float(scan_time_tgt)
                    if scan_time_tgt is not None
                    else np.nan,
                    "scan_coadds": float(scan_coadds_tgt)
                    if scan_coadds_tgt is not None
                    else np.nan,
                    "_vector": rad_tgt,
                }
            )

            # Add reference candidates for reflectance interpolation:
            # - R uses ref_time
            # - Tp uses tgt_time (target-side)
            references_by_group[group_key].append(
                {
                    "scan_id": scan_id,
                    "scan_type": "R",
                    "time": ref_time,
                    "rad": rad_ref,
                }
            )
            if target_type == "Tp":
                references_by_group[group_key].append(
                    {
                        "scan_id": scan_id,
                        "scan_type": "Tp",
                        "time": tgt_time,
                        "rad": rad_tgt,
                    }
                )
            if target_type == "T":
                targets_by_group[group_key].append(
                    {
                        "scan_id": scan_id,
                        "time": tgt_time,
                        "rad": rad_tgt,
                        "sig_file": sig_path.name,
                        "meta": radiance_rows[
                            -1
                        ],  # target-side meta dict (includes _vector)
                    }
                )

    if native_wl is None:
        raise ValueError(f"No SIG data read for site {site}")

    # Build wide columns
    wl_cols_native = make_wl_columns(
        native_wl, config.SPECTROMETER["wavelength"]["column_format"]
    )

    # Radiance dataframe
    rad_df = pd.DataFrame(radiance_rows)
    # Create UTC columns and local_date
    tz = config.SPECTROMETER["timezone"]
    rad_df["acquisition_time_local"] = pd.to_datetime(
        rad_df["acquisition_time_local"], errors="coerce"
    )
    rad_df["acquisition_time_local"] = rad_df["acquisition_time_local"].dt.tz_localize(
        tz, nonexistent="shift_forward", ambiguous="NaT"
    )
    rad_df["acquisition_time_utc"] = rad_df["acquisition_time_local"].dt.tz_convert(
        "UTC"
    )
    rad_df["local_date"] = rad_df["acquisition_time_local"].dt.date

    # Expand vector into wide columns
    mat = np.vstack(rad_df["_vector"].to_numpy())
    spec_df = pd.DataFrame(mat, columns=wl_cols_native)
    rad_df = pd.concat([rad_df.drop(columns=["_vector"]), spec_df], axis=1)

    # Reflectance native (T scans only)
    panel_interp = interp_panel_to_wavelengths(panel, native_wl)

    refl_rows = []
    for (station_id, round_num), t_list in targets_by_group.items():
        refs = [
            r
            for r in references_by_group[(station_id, round_num)]
            if r["time"] is not None
        ]
        refs.sort(key=lambda r: r["time"])

        if len(refs) == 0:
            raise ValueError(f"No references found for {station_id} round {round_num}")

        ref_times = [r["time"] for r in refs]
        ref_rads = [r["rad"] for r in refs]
        ref_ids = [r["scan_id"] for r in refs]
        ref_types = [r["scan_type"] for r in refs]

        for t in t_list:
            t_time = t["time"]
            if t_time is None or pd.isna(t_time):
                raise ValueError(
                    f"Missing target time for {station_id} round {round_num} scan {t['scan_id']}"
                )

            # find insertion point
            idx = np.searchsorted(ref_times, t_time)
            before_idx = idx - 1
            after_idx = idx

            warning_flag = False
            warning_msg = None

            if before_idx < 0 and after_idx >= len(refs):
                raise ValueError(
                    f"No usable references for target scan {t['scan_id']} (should not happen)"
                )
            elif before_idx < 0:
                # only after available
                before = None
                after = refs[after_idx]
                ref_mode = "after_only"
                warning_flag = True
                warning_msg = "Missing before reference; used nearest after reference"
                R_interp = after["rad"]
                before_id = None
                after_id = after["scan_id"]
                frac = None
            elif after_idx >= len(refs):
                # only before available
                before = refs[before_idx]
                after = None
                ref_mode = "before_only"
                warning_flag = True
                warning_msg = "Missing after reference; used nearest before reference"
                R_interp = before["rad"]
                before_id = before["scan_id"]
                after_id = None
                frac = None
            else:
                before = refs[before_idx]
                after = refs[after_idx]
                t0 = before["time"]
                t1 = after["time"]
                if t0 == t1:
                    ref_mode = "interp"
                    frac = 0.0
                    R_interp = before["rad"]
                    before_id = before["scan_id"]
                    after_id = after["scan_id"]
                else:
                    ref_mode = "interp"
                    frac = (t_time - t0).total_seconds() / (t1 - t0).total_seconds()
                    R_interp = before["rad"] + frac * (after["rad"] - before["rad"])
                    before_id = before["scan_id"]
                    after_id = after["scan_id"]

            rel = t["rad"] / R_interp
            abs_refl = rel * panel_interp

            if np.isnan(panel_interp).any():
                warning_flag = True
                warning_msg = (
                    warning_msg + " | " if warning_msg else ""
                ) + "Panel curve out-of-range produced NaNs"

            # Build output row
            base_meta = t["meta"].copy()
            base_meta.pop("_vector", None)

            out = {
                "site": site,
                "station_id": station_id,
                "round": int(round_num),
                "scan_id": int(t["scan_id"]),
                "scan_type": "T",
                "sig_file": t["sig_file"],
                "acquisition_time_local": t_time,
                "latitude_deg": base_meta.get("latitude_deg"),
                "longitude_deg": base_meta.get("longitude_deg"),
                "sun_zenith_deg": base_meta.get("sun_zenith_deg"),
                "sun_azimuth_deg": base_meta.get("sun_azimuth_deg"),
                "scan_method": base_meta.get("scan_method"),
                "scan_time_s": base_meta.get("scan_time_s"),
                "scan_coadds": base_meta.get("scan_coadds"),
                "reference_mode": ref_mode,
                "before_reference_scan_id": before_id,
                "after_reference_scan_id": after_id,
                "interpolation_fraction": frac,
                "warning_flag": bool(warning_flag),
                "warning_message": warning_msg,
                "_vector": abs_refl,
            }
            refl_rows.append(out)

    refl_df = pd.DataFrame(refl_rows)
    refl_df["acquisition_time_local"] = pd.to_datetime(
        refl_df["acquisition_time_local"], errors="coerce"
    )
    refl_df["acquisition_time_local"] = refl_df[
        "acquisition_time_local"
    ].dt.tz_localize(tz, nonexistent="shift_forward", ambiguous="NaT")
    refl_df["acquisition_time_utc"] = refl_df["acquisition_time_local"].dt.tz_convert(
        "UTC"
    )
    refl_df["local_date"] = refl_df["acquisition_time_local"].dt.date

    mat = np.vstack(refl_df["_vector"].to_numpy())
    spec_df = pd.DataFrame(mat, columns=wl_cols_native)
    refl_native = pd.concat([refl_df.drop(columns=["_vector"]), spec_df], axis=1)

    # Check than station_id + round are unique in summary tables and have 20 or 30 scans in total
    scan_counts = (
        refl_native.groupby(["station_id", "round"])
        .size()
        .reset_index(name="scan_count")
    )
    # Check only stations that are not 'grey' or 'black'
    if (
        not scan_counts.loc[
            ~scan_counts["station_id"].isin(["grey", "black"]), "scan_count"
        ]
        .isin([20, 30])
        .all()
    ):
        # if not scan_counts["scan_count"].isin([20, 30]).all():
        raise ValueError(
            "Some station_id + round combinations do not have 20 or 30 scans"
        )

    # Add sample_type == R for all of the scans for stations_id and round that has a total of 20 scans
    # Use sample_type == S for the 10 first scans for stations_id and round that has a total of 30 scans, the 20 next scans will be sample_type == R

    group_cols = ["station_id", "round"]

    # Number of scans in each station/round group
    scan_count = refl_native.groupby(group_cols)["scan_id"].transform("size")

    # Position of each scan after ordering by scan_id within its group:
    # 1 = first scan, 2 = second scan, etc.
    scan_order = (
        refl_native.groupby(group_cols)["scan_id"]
        .rank(method="first", ascending=True)
        .astype("int64")
    )

    # Groups with 20 scans: all scans are R
    refl_native.loc[scan_count.eq(20), "sample_type"] = "R"

    # Groups with 30 scans: first 10 scans are S, remaining 20 are R
    groups_with_30 = scan_count.eq(30)

    refl_native.loc[
        groups_with_30 & scan_order.le(10),
        "sample_type",
    ] = "S"

    refl_native.loc[
        groups_with_30 & scan_order.gt(10),
        "sample_type",
    ] = "R"

    # Give sample type to 'O' for 'grey' or 'black' entries
    refl_native.loc[
        refl_native["station_id"].isin(["grey", "black"]), "sample_type"
    ] = "O"

    # Group by consecutive pair through a pair id
    pair_group_cols = ["station_id", "round", "sample_type"]
    sample_type_order = (
        refl_native.groupby(pair_group_cols)["scan_id"]
        .rank(method="first", ascending=True)
        .astype("int64")
    )
    refl_native["pair_id"] = ((sample_type_order - 1) // 2).astype("Int64")

    # Place 'sample_type' and 'pair_id' columns after 'scan_type' column
    cols = refl_native.columns.tolist()
    if "scan_type" in cols:
        scan_type_index = cols.index("scan_type")
        cols.insert(scan_type_index + 1, cols.pop(cols.index("sample_type")))
        cols.insert(scan_type_index + 2, cols.pop(cols.index("pair_id")))
        refl_native = refl_native[cols]

    # Common grid resample
    cg = config.SPECTROMETER["wavelength"]["common_grid"]
    common_wl = np.arange(
        cg["start_nm"], cg["stop_nm"] + 0.5 * cg["step_nm"], cg["step_nm"], dtype=float
    )
    wl_cols_common = make_wl_columns(common_wl, config.SPECTROMETER["wavelength"]["column_format"])

    # linear interpolate each scan spectrum onto common_wl; NaN outside native_wl range
    native_x = native_wl
    native_min, native_max = float(np.nanmin(native_x)), float(np.nanmax(native_x))

    common_vectors = []
    for i in range(len(refl_native)):
        y = refl_native[wl_cols_native].iloc[i].to_numpy(dtype=float)
        y_common = np.interp(common_wl, native_x, y, left=np.nan, right=np.nan)
        common_vectors.append(y_common)

    common_mat = np.vstack(common_vectors)
    common_spec = pd.DataFrame(common_mat, columns=wl_cols_common)
    refl_common = pd.concat(
        [refl_native.drop(columns=wl_cols_native), common_spec], axis=1
    )

    # Summaries by station_id + round + ...
    group_cols = ["site", "station_id", "round", "sample_type"]

    summary_native = summarize_by_group_stat_rows(
        refl_native, group_cols, wl_cols_native
    )
    summary_common = summarize_by_group_stat_rows(
        refl_common, group_cols, wl_cols_common
    )

    # Save files
    compression = 'zstd'

    # return SiteProducts(
    #     radiance_native=rad_df,
    #     refl_native_scans=refl_native,
    #     refl_common_scans=refl_common,
    #     summary_native=summary_native,
    #     summary_common=summary_common,
    #     native_wavelengths=native_wl,
    #     common_wavelengths=common_wl,

    # if cfg["outputs"]["write_radiance_native"]: # Always true now
    p = out_dir / f"{site}_radiance_native_wavelengths.parquet"
    # rad_df.to_parquet(p, index=False, compression=compression)
    write_df_to_parquet(rad_df, p, index=False, compression=compression)

    p = out_dir / f"{site}_reflectance_native_wavelengths_scans.parquet"
    write_df_to_parquet(refl_native, p, index=False, compression=compression)

    p = out_dir / f"{site}_reflectance_common_wavelengths_scans.parquet"
    write_df_to_parquet(refl_common, p, index=False, compression=compression)

    p = out_dir / f"{site}_station_summary_native_wavelengths.parquet"
    write_df_to_parquet(summary_native, p, index=False, compression=compression)

    p = out_dir / f"{site}_station_summary_common_wavelengths.parquet"
    write_df_to_parquet(summary_common, p, index=False, compression=compression)

    print(f"Finished processing site {site}!")


Finished processing site brunswick!
Finished processing site westham!


# QAQC (TBD)